# Семинар: Эконометрика в Machine Learning

## Цель семинара

Научиться проверять корректность построенной линейной регрессии и понимать, когда её коэффициентам и доверительным интервалам можно верить.

В машинном обучении мы постоянно строим регрессионные модели. Но как мы можем быть уверегы, что её выводы статистически обоснованы.

## План семинара

1. F-тест и сравнение вложенных моделей
2. Гетероскедастичность и робастные ошибки
3. Проверка нормальности остатков
4. Теорема Фриша-Во-Ловелла (FWL)

---

# Блок 1. F-тест: Сравнение вложенных моделей

## Постановка задачи

В машинном обучении мы постоянно добавляем признаки в модель. Но в прошлый семинар мы видели, что каждый новый признак это риск переобучения. Как нам понять, что новый признак статистически значимо улучшает модель, а не просто подстраивается под шум?

Представим: мы обучили линейную регрессию с 3 признаками. $R^2 = 0.64$. Потом добавили ещё 2 признака. $R^2$ стал $0.65$. Улучшилась ли модель? На первый взгляд почти нет, но с точки зрения статистики?

Для этого существует F-тест (тест Фишера). Он сравнивает две вложенные модели: Неограниченную (UR) (полную модель со всеми признаками) и Ограниченную (R) (модель с частью признаков (или с линейными ограничениями на коэффициенты))

F-статистика измеряет, *насколько сильно выросла ошибка (RSS), когда мы "запретили" части модели быть свободными*. Если $RSS_R \gg RSS_{UR}$ — ограничения вредны, признаки нужны. Если $RSS_R \approx RSS_{UR}$ — ограничения безболезненны, признаки можно выкинуть.

## Формула

$$F = \frac{(RSS_R - RSS_{UR}) / r}{RSS_{UR} / (n - p)} \sim F(r, n-p)$$

где:
- $r$ — число ограничений (сколько коэффициентов мы «запретили»)
- $p$ — число параметров в неограниченной модели (включая константу)
- $n$ — размер выборки

## ML-интерпретация

F-тест отвечает на вопрос: «Можно ли наложить линейные ограничения на коэффициенты регрессии без существенной потери качества?»

Частные случаи:
- «Можно ли приравнять $\beta_3 = \beta_4 = 0$?» — отбор признаков
- «Можно ли приравнять все $\beta$, кроме константы, к нулю?» — проверка значимости модели в целом
- «Можно ли приравнять $\beta_1 + \beta_2 = 1$?» — проверка экономической теории

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

np.random.seed(42)
n = 500

# Генерируем данные: Y зависит от X1, X2, X3, но НЕ от X4, X5
X1 = np.random.normal(0, 1, n)
X2 = np.random.normal(0, 1, n)
X3 = np.random.normal(0, 1, n)
X4 = np.random.normal(0, 1, n)  # "шумовой" признак
X5 = np.random.normal(0, 1, n)  # "шумовой" признак

# Истинная модель: Y = 2 + 3*X1 - 1.5*X2 + 0.8*X3 + noise
Y = 2 + 3*X1 - 1.5*X2 + 0.8*X3 + np.random.normal(0, 2, n)

data = pd.DataFrame({'Y': Y, 'X1': X1, 'X2': X2, 'X3': X3, 'X4': X4, 'X5': X5})

# Неограниченная модель (все 5 признаков)
model_ur = smf.ols('Y ~ X1 + X2 + X3 + X4 + X5', data=data).fit()
RSS_UR = np.sum(model_ur.resid**2)

# Ограниченная модель (только X1, X2, X3 — т.е. гипотеза H0: β4 = β5 = 0)
model_r = smf.ols('Y ~ X1 + X2 + X3', data=data).fit()
RSS_R = np.sum(model_r.resid**2)

# F-статистика
p = model_ur.df_model + 1  # число параметров в UR (включая константу)
r = 2  # число ограничений (β4 = 0 и β5 = 0)
F_stat = ((RSS_R - RSS_UR) / r) / (RSS_UR / (n - p))
p_value = 1 - stats.f.cdf(F_stat, r, n - p)

# === ВИЗУАЛИЗАЦИЯ ===
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Левый: сравнение RSS
bars = axes[0].bar(['Ограниченная\n(3 признака)', 'Неограниченная\n(5 признаков)'],
                   [RSS_R, RSS_UR], color=['lightcoral', 'lightgreen'],
                   edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('RSS (сумма квадратов остатков)', fontsize=11)
axes[0].set_title('Сравнение качества моделей', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, [RSS_R, RSS_UR]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{val:.1f}', ha='center', fontsize=11, fontweight='bold')

# Центральный: F-распределение с отмеченной статистикой
x_f = np.linspace(0, 5, 500)
y_f = stats.f.pdf(x_f, r, n - p)
axes[1].plot(x_f, y_f, 'b-', linewidth=2, label=f'F({r}, {n-p})')
axes[1].axvline(F_stat, color='red', linestyle='--', linewidth=2.5,
               label=f'F-статистика = {F_stat:.3f}')
axes[1].axvline(stats.f.ppf(0.95, r, n - p), color='green', linestyle=':',
               linewidth=2, label=f'Критическое значение (α=0.05) = {stats.f.ppf(0.95, r, n-p):.3f}')
axes[1].fill_between(x_f, y_f, where=(x_f >= stats.f.ppf(0.95, r, n-p)),
                    color='red', alpha=0.2, label='Зона отвержения H₀')
axes[1].set_xlabel('Значение F', fontsize=11)
axes[1].set_ylabel('Плотность', fontsize=11)
axes[1].set_title(f'F-тест\np-value = {p_value:.4e}', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

# Правый: коэффициенты UR модели с доверительными интервалами
coefs = model_ur.params[1:]  # без константы
ci = model_ur.conf_int().loc[coefs.index]
y_pos = np.arange(len(coefs))

# Создаем маску для значимых коэффициентов
sig_mask = (ci.iloc[:, 0] * ci.iloc[:, 1]) > 0  # Если произведение > 0, знаки одинаковые (0 не входит)

for mask, col in [(sig_mask, 'red'), (~sig_mask, 'gray')]:
    if mask.any():  # Проверяем, есть ли вообще точки в этой группе
        sub_coefs = coefs[mask]
        sub_ci = ci[mask]
        axes[2].errorbar(
            sub_coefs.values, y_pos[mask],
            xerr=[sub_coefs.values - sub_ci.iloc[:, 0], sub_ci.iloc[:, 1] - sub_coefs.values],
            fmt='o', color='black', ecolor=col, capsize=5, linewidth=2, markersize=8
        )

axes[2].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[2].set_yticks(y_pos)
axes[2].set_yticklabels(coefs.index)
axes[2].set_xlabel('Значение коэффициента', fontsize=11)
axes[2].set_title('Коэффициенты полной модели\n(красные ДИ не содержат 0 → значимы)',
                 fontsize=12, fontweight='bold')
axes[2].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print(f"RSS ограниченной модели: {RSS_R:.2f}")
print(f"RSS неограниченной модели: {RSS_UR:.2f}")
print(f"F-статистика: {F_stat:.4f}")
print(f"p-value: {p_value:.4e}")
print(f"Вывод: {'H0 ОТВЕРГАЕТСЯ — признаки X4, X5 значимы' if p_value < 0.05 else 'H0 НЕ ОТВЕРГАЕТСЯ — признаки X4, X5 можно выкинуть'}")

## Что мы увидели на графике

1. Левый график (сравнение RSS): RSS почти не вырос, когда мы выкинули X4 и X5. Это первый сигнал, что признаки не нужны.

2. Центральный график (F-распределение): Красная пунктирная линия (наша F-статистика) находится далеко слева от зелёной критической границы. Мы не попадаем в красную зону отвержения. Значит, p-value большой, $H_0$ не отвергаем.

3. Правый график (коэффициенты): X4 и X5 серые, их доверительные интервалы содержат ноль. Они статистически не значимы. А вот X1, X2, X3 — красные, их ДИ не содержат ноль. Это и есть "сигнал".

## Ключевой вывод для ML

F-тест — это строгий статистический аналог вашей интуиции «а нужен ли этот признак?». Он работает для вложенных моделей (когда одна модель — частный случай другой). Для невложенных моделей используйте AIC/BIC или кросс-валидацию.

---

# Блок 2. Гетероскедастичность: Когда доверительным интервалам нельзя верить

## Критическое допущение МНК

Мы построили линейную регрессию. Коэффициенты выглядят значимыми, доверительные интервалы узкие. Можно идти к бизнесу?

Есть одно критическое допущение МНК, о котором часто забывают: гомоскедастичность — постоянство дисперсии ошибок.

Если дисперсия ошибок растёт с ростом X (например, богатые клиенты тратят деньги с большим разбросом, чем бедные), то коэффициенты всё ещё несмещённые (оценка верная), но стандартные ошибки неверные → доверительные интервалы и p-value лгут о своей значимости.

Это называется гетероскедастичность. В ML она встречается постоянно: в финансовых данных, в данных с «тяжёлыми хвостами», в данных с разными масштабами.

## Диагностика

1. Визуально: scatter plot остатков на X. Если он выглядит, как «воронку» или «веер», гетероскедастичность налицо.
2. Формально: тест Бройша-Пагана (Breusch-Pagan) или тест Голдфелда-Квандта.

## Лечение

1. Робастные стандартные ошибки (HC) — исправляют стандартные ошибки, не меняя коэффициенты.
2. WLS (взвешенный МНК) — если известна структура гетероскедастичности.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms

np.random.seed(42)
n = 200

# Генерируем данные С гетероскедастичностью: Var(ε) растёт с X
X = np.random.uniform(1, 10, n)
eps = np.random.normal(0, X, n)  # дисперсия = X²
Y = 2 + 1.5*X + eps

data = pd.DataFrame({'X': X, 'Y': Y})

# Обычный МНК (игнорирует гетероскедастичность)
model_ols = smf.ols('Y ~ X', data=data).fit()

# Тест Бройша-Пагана
bp_stat, bp_pval, _, _ = sms.het_breuschpagan(model_ols.resid, model_ols.model.exog)

# Робастные стандартные ошибки (HC3) — передаем сразу в fit()
model_hc = smf.ols('Y ~ X', data=data).fit(cov_type='HC3')

# === ВИЗУАЛИЗАЦИЯ ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Верхний левый:.scatter с линией регрессии
axes[0, 0].scatter(X, Y, alpha=0.5, s=20, color='steelblue')
x_line = np.linspace(X.min(), X.max(), 100)
y_line = model_ols.params.iloc[0] + model_ols.params.iloc[1] * x_line
axes[0, 0].plot(x_line, y_line, 'r-', linewidth=2.5, label='МНК-регрессия')
axes[0, 0].set_xlabel('X', fontsize=11)
axes[0, 0].set_ylabel('Y', fontsize=11)
axes[0, 0].set_title('Данные с гетероскедастичностью\n(разброс растёт с X)',
                     fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Верхний правый: остатки vs X (визуальная диагностика)
axes[0, 1].scatter(X, model_ols.resid, alpha=0.5, s=20, color='darkorange')
axes[0, 1].axhline(0, color='black', linestyle='--', linewidth=1.5)
axes[0, 1].set_xlabel('X', fontsize=11)
axes[0, 1].set_ylabel('Остатки (residuals)', fontsize=11)
axes[0, 1].set_title('Диагностика: остатки vs X\n(видна "воронка" → гетероскедастичность)',
                     fontsize=12, fontweight='bold')
axes[0, 1].grid(alpha=0.3)

# Нижний левый: сравнение стандартных ошибок
coefs_names = ['Intercept', 'X']
se_ols = [model_ols.bse.iloc[0], model_ols.bse.iloc[1]]
se_hc = [model_hc.bse.iloc[0], model_hc.bse.iloc[1]]

x_pos = np.arange(len(coefs_names))
width = 0.35
bars1 = axes[1, 0].bar(x_pos - width/2, se_ols, width, label='Обычные SE (МНК)',
                       color='lightcoral', edgecolor='black')
bars2 = axes[1, 0].bar(x_pos + width/2, se_hc, width, label='Робастные SE (HC3)',
                       color='lightgreen', edgecolor='black')
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels(coefs_names)
axes[1, 0].set_ylabel('Стандартная ошибка', fontsize=11)
axes[1, 0].set_title('Сравнение стандартных ошибок\n(робастные обычно БОЛЬШЕ)',
                     fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(axis='y', alpha=0.3)

# Нижний правый: p-value теста Бройша-Пагана
color = 'red' if bp_pval < 0.05 else 'green'
axes[1, 1].bar(['p-value теста BP'], [bp_pval], color=color, alpha=0.7,
              edgecolor='black', linewidth=2)
axes[1, 1].axhline(0.05, color='red', linestyle='--', linewidth=2, label='α = 0.05')
axes[1, 1].set_ylim(0, max(bp_pval * 1.5, 0.1))
axes[1, 1].set_ylabel('p-value', fontsize=11)
axes[1, 1].set_title(f'Тест Бройша-Пагана\nСтатистика = {bp_stat:.2f}',
                     fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

conclusion = "⚠️ ОБНАРУЖЕНА ГЕТЕРОСКЕДАСТИЧНОСТЬ\nОбычные SE и p-value ненадёжны\n→ Используйте робастные SE (HC)" if bp_pval < 0.05 else "✓ Гетероскедастичность не обнаружена"
axes[1, 1].text(0, 0.05, conclusion,
               ha='center', va='center', fontsize=10, fontweight='bold', color=color,
               bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor=color, linewidth=2))

plt.tight_layout()
plt.show()

print(f"Тест Бройша-Пагана: статистика = {bp_stat:.3f}, p-value = {bp_pval:.4e}")
print(f"\nСравнение p-value для коэффициента при X:")
print(f"  Обычный МНК: {model_ols.pvalues.iloc[1]:.4e}")
print(f"  Робастный HC3: {model_hc.pvalues.iloc[1]:.4e}")

## Что мы увидели на графике

1. Верхний левый (scatter plot): Видим, как точки «разбегаются» вправо. Это и есть гетероскедастичность — при больших X разброс Y намного больше.

2. Верхний правый (остатки vs X): Классический диагностический график — остатки против X. Если видим «воронку» или «веер» — гетероскедастичность налицо.

3. Нижний левый (сравнение SE): Робастные стандартные ошибки (зелёные) обычно больше обычных (красных). Это значит, что обычные МНК-оценки были слишком оптимистичны.

4. Нижний правый (тест Бройша-Пагана): Формально подтверждает: p-value < 0.05, гетероскедастичность есть.

## Ключевой вывод для ML

Всегда проверяйте гетероскедастичность, если строите доверительные интервалы для коэффициентов. Если она есть — используйте `ols('Y ~ X', data=data).fit(cov_type='HC3')` в statsmodels. Это одна строчка кода, но она спасает от ложных выводов.

---

# Блок 3. Проверка нормальности остатков: Адекватна ли модель?

## Зачем проверять нормальность?

Даже если гетероскедастичности нет, остаётся вопрос: нормальны ли остатки? Это важно для:
- Корректности доверительных интервалов
- Корректности p-value
- Применимости t- и F-тестов

В ML мы часто имеем дело с «тяжёлыми хвостами» (выбросами), скошенными распределениями, мультимодальностью. Всё это нарушает нормальность.

## Тесты нормальности

Есть много тестов:
- Колмогорова-Смирнова (KS) — сравнивает эмпирическое распределение с нормальным
- Андерсона-Дарлинга (AD) — более чувствителен к хвостам
- Харке-Бера (JB) — основан на асимметрии и эксцессе
- Шапиро-Уилка — один из самых мощных для малых выборок

Но у них разная "мощность" — способность обнаружить отклонение от нормальности. Разберёмся, какой когда применять.

Мы проведём эксперимент: сгенерируем данные с разными типами ошибок (нормальные, экспоненциальные, t-распределение) и посмотрим, какой тест лучше обнаруживает отклонение.

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.stats.diagnostic as smd
import statsmodels.stats.stattools as smst

np.random.seed(42)


def check_power(sample_sizes, num_samples=500, alpha=0.05, error_type='exponential'):
    results = {'KS': {}, 'AD': {}, 'JB': {}}

    for n in sample_sizes:
        accepted_ks = accepted_ad = accepted_jb = 0

        for _ in range(num_samples):
            X = np.random.normal(0, 1, n)

            if error_type == 'exponential':
                eps = np.random.exponential(1, n) - 1
            elif error_type == 'normal':
                eps = np.random.normal(0, 1, n)
            else:
                eps = np.random.standard_t(3, n)

            Y = 2 + 1.5 * X + eps

            # БЫСТРЫЙ МНК НА NUMPY (без DataFrame и statsmodels)
            X_mat = np.column_stack((np.ones(n), X))
            beta, _, _, _ = np.linalg.lstsq(X_mat, Y, rcond=None)
            res = Y - X_mat @ beta

            # Тесты (исправлена распаковка normal_ad)
            _, p_ks = smd.kstest_normal(res)
            _, p_ad = smd.normal_ad(res)
            _, p_jb = smst.jarque_bera(res)[:2]

            if p_ks > alpha: accepted_ks += 1
            if p_ad > alpha: accepted_ad += 1
            if p_jb > alpha: accepted_jb += 1

        results['KS'][n] = accepted_ks / num_samples
        results['AD'][n] = accepted_ad / num_samples
        results['JB'][n] = accepted_jb / num_samples

    return results


# Проверка мощности на разных размерах выборки
sample_sizes = [30, 50, 100, 200, 500, 1000]
power_exp = check_power(sample_sizes, error_type='exponential')
power_norm = check_power(sample_sizes, error_type='normal')
power_t = check_power(sample_sizes, error_type='t')

# === ВИЗУАЛИЗАЦИЯ ===
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Левый: экспоненциальные ошибки (сильное отклонение)
axes[0].plot(sample_sizes, [power_exp['KS'][n] for n in sample_sizes],
             'o-', label='KS', linewidth=2, markersize=8)
axes[0].plot(sample_sizes, [power_exp['AD'][n] for n in sample_sizes],
             's-', label='Anderson-Darling', linewidth=2, markersize=8)
axes[0].plot(sample_sizes, [power_exp['JB'][n] for n in sample_sizes],
             '^-', label='Jarque-Bera', linewidth=2, markersize=8)
axes[0].axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='Уровень значимости α=0.05')
axes[0].set_xlabel('Размер выборки n', fontsize=11)
axes[0].set_ylabel('Доля принятых H₀ (из 500)', fontsize=11)
axes[0].set_title('Экспоненциальные ошибки\n(сильное отклонение от нормальности)',
                  fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0, 1.05)

# Центральный: нормальные ошибки (H₀ верна)
axes[1].plot(sample_sizes, [power_norm['KS'][n] for n in sample_sizes],
             'o-', label='KS', linewidth=2, markersize=8)
axes[1].plot(sample_sizes, [power_norm['AD'][n] for n in sample_sizes],
             's-', label='Anderson-Darling', linewidth=2, markersize=8)
axes[1].plot(sample_sizes, [power_norm['JB'][n] for n in sample_sizes],
             '^-', label='Jarque-Bera', linewidth=2, markersize=8)
axes[1].axhline(0.95, color='green', linestyle='--', linewidth=1.5, label='Ожидаемая доля (1-α)')
axes[1].set_xlabel('Размер выборки n', fontsize=11)
axes[1].set_ylabel('Доля принятых H₀ (из 500)', fontsize=11)
axes[1].set_title('Нормальные ошибки\n(H₀ верна — тесты должны принимать)',
                  fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 1.05)

# Правый: t-распределение (тяжёлые хвосты)
axes[2].plot(sample_sizes, [power_t['KS'][n] for n in sample_sizes],
             'o-', label='KS', linewidth=2, markersize=8)
axes[2].plot(sample_sizes, [power_t['AD'][n] for n in sample_sizes],
             's-', label='Anderson-Darling', linewidth=2, markersize=8)
axes[2].plot(sample_sizes, [power_t['JB'][n] for n in sample_sizes],
             '^-', label='Jarque-Bera', linewidth=2, markersize=8)
axes[2].axhline(0.05, color='red', linestyle='--', linewidth=1.5, label='Уровень значимости α=0.05')
axes[2].set_xlabel('Размер выборки n', fontsize=11)
axes[2].set_ylabel('Доля принятых H₀ (из 500)', fontsize=11)
axes[2].set_title('t-распределение (df=3)\n(тяжёлые хвосты)',
                  fontsize=12, fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3)
axes[2].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print("Интерпретация:")
print("Левый график: чем НИЖЕ линия — тем ВЫШЕ мощность теста (лучше обнаруживает отклонение)")
print("Центральный график: линии должны быть около 0.95 (тесты не должны отвергать верную H₀)")
print("Правый график: t-распределение сложнее обнаружить, но AD и JB справляются лучше KS")

## Что мы увидели на графиках

1. Левый график (экспоненциальные ошибки): Все три теста хорошо обнаруживают отклонение. Но Anderson-Darling (красный) и Jarque-Bera (зелёный) быстрее и надёжнее. KS (синий) на малых выборках может «пропустить» отклонение.

2. Центральный график (нормальные ошибки): Когда $H_0$ верна, все тесты принимают её примерно в 95% случаев (как и должно быть при $\alpha=0.05$). Это проверка калибровки тестов.

3. Правый график (t-распределение): Тяжёлые хвосты сложны в поиске. KS-тест плохо их ловит. AD и JB могут справится лучше. Вывод: для ML-данных с выбросами используйте AD или JB.

## Ключевой вывод для ML

Не доверяйте одному тесту. Всегда смотрите на QQ-plot (квантиль-квантильный график) визуально. Если точки сильно отклоняются от прямой, это первый признак того, что нормальности нет, и нужно либо трансформировать целевую переменную (логарифмировать, Box-Cox), либо использовать робастные методы.

---

# Блок 4. Теорема FWL: Что на самом деле измеряет коэффициент?

## Интерпретация коэффициентов

Когда мы видим коэффициент $\beta_1 = 2.5$ при признаке $X_1$ в множественной регрессии, что это значит?

<details>
<summary><b>Неправильный ответ:</b></summary>

«При увеличении $X_1$ на 1, $Y$ увеличивается на 2.5».

</details>

<details>
<summary><b>Правильный ответ:</b></summary>

 «При увеличении $X_1$ на 1, $Y$ увеличивается на 2.5, при условии, что все остальные признаки фиксированы».

Но что значит «при условии, что другие признаки фиксированы»? Это же невозможно в реальных данных!
</details>


## Теорема Фриша-Во-Ловелла (FWL)

Теорема Фриша-Во-Ловелла (FWL) даёт элегантный ответ: коэффициент $\beta_1$ можно получить в три шага:

Шаг 1. Регрессируем $Y$ на все признаки, **КРОМЕ** $X_1$ → получаем остатки $r_Y$

Шаг 2. Регрессируем $X_1$ на все признаки, **КРОМЕ** $X_1$ → получаем остатки $r_{X_1}$

Шаг 3. Регрессируем $r_Y$ на $r_{X_1}$ → коэффициент = $\beta_1$

Это значит: $\beta_1$ измеряет влияние "чистого" $X_1$ (очищенного от влияния других признаков) на "чистый" $Y$ (очищенный от влияния других признаков).

## Связь с ML

В ML это напрямую связано с Partial Dependence Plots (показывают «чистый» эффект признака), SHAP-значениями (разлагают предсказание на вклад каждого признака) и Feature importance** в деревьях (но там есть свои нюансы)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

np.random.seed(42)
n = 300

# Генерируем коррелированные признаки
X1 = np.random.normal(0, 1, n)
X2 = 0.7*X1 + np.random.normal(0, 0.5, n)  # X2 коррелирует с X1
X3 = np.random.normal(0, 1, n)

# Y зависит от всех трёх
Y = 2 + 3*X1 - 1.5*X2 + 0.8*X3 + np.random.normal(0, 1, n)

data = pd.DataFrame({'Y': Y, 'X1': X1, 'X2': X2, 'X3': X3})

# Обычная множественная регрессия
model_full = smf.ols('Y ~ X1 + X2 + X3', data=data).fit()
beta1_full = model_full.params['X1']

# === ТЕОРЕМА FWL: пошагово ===

# Шаг 1: Регрессируем Y на X2, X3 (всё, кроме X1) → остатки r_Y
model_y = smf.ols('Y ~ X2 + X3', data=data).fit()
r_Y = model_y.resid

# Шаг 2: Регрессируем X1 на X2, X3 → остатки r_X1
model_x1 = smf.ols('X1 ~ X2 + X3', data=data).fit()
r_X1 = model_x1.resid

# Шаг 3: Регрессируем r_Y на r_X1 → коэффициент = beta1
model_fwl = smf.ols('r_Y ~ r_X1', data=pd.DataFrame({'r_Y': r_Y, 'r_X1': r_X1})).fit()
beta1_fwl = model_fwl.params['r_X1']

# === ВИЗУАЛИЗАЦИЯ ===
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Верхний левый: исходные данные (Y vs X1)
axes[0, 0].scatter(X1, Y, alpha=0.4, s=20, color='steelblue')
z = np.polyfit(X1, Y, 1)
p = np.poly1d(z)
x_line = np.linspace(X1.min(), X1.max(), 100)
axes[0, 0].plot(x_line, p(x_line), 'r-', linewidth=2.5, label=f'Простая регрессия: β = {z[0]:.3f}')
axes[0, 0].set_xlabel('X1', fontsize=11)
axes[0, 0].set_ylabel('Y', fontsize=11)
axes[0, 0].set_title('Исходные данные: Y vs X1\n(без учёта X2, X3)',
                     fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Верхний правый: FWL — "очищенные" данные
axes[0, 1].scatter(r_X1, r_Y, alpha=0.4, s=20, color='darkorange')
z_fwl = np.polyfit(r_X1, r_Y, 1)
p_fwl = np.poly1d(z_fwl)
x_line_fwl = np.linspace(r_X1.min(), r_X1.max(), 100)
axes[0, 1].plot(x_line_fwl, p_fwl(x_line_fwl), 'r-', linewidth=2.5,
               label=f'FWL регрессия: β = {z_fwl[0]:.3f}')
axes[0, 1].set_xlabel('Остатки X1 (очищенные от X2, X3)', fontsize=11)
axes[0, 1].set_ylabel('Остатки Y (очищенные от X2, X3)', fontsize=11)
axes[0, 1].set_title('Теорема FWL: "чистый" эффект X1 на Y\n(после удаления влияния X2, X3)',
                     fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Нижний левый: сравнение коэффициентов
bars = axes[1, 0].bar(['Простая регрессия\n(Y ~ X1)', 'Множественная регрессия\n(Y ~ X1 + X2 + X3)',
                       'FWL регрессия\n(r_Y ~ r_X1)'],
                      [z[0], beta1_full, beta1_fwl],
                      color=['lightcoral', 'lightgreen', 'lightblue'],
                      edgecolor='black', linewidth=1.5)
axes[1, 0].axhline(0, color='black', linestyle='-', linewidth=1)
axes[1, 0].set_ylabel('Коэффициент при X1', fontsize=11)
axes[1, 0].set_title('Сравнение оценок коэффициента β₁', fontsize=12, fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)
for bar, val in zip(bars, [z[0], beta1_full, beta1_fwl]):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                   f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

# Нижний правый: корреляция признаков
corr_matrix = data[['X1', 'X2', 'X3']].corr()
im = axes[1, 1].imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
axes[1, 1].set_xticks(range(3))
axes[1, 1].set_yticks(range(3))
axes[1, 1].set_xticklabels(['X1', 'X2', 'X3'])
axes[1, 1].set_yticklabels(['X1', 'X2', 'X3'])
for i in range(3):
    for j in range(3):
        axes[1, 1].text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                       ha='center', va='center', fontsize=12, fontweight='bold',
                       color='white' if abs(corr_matrix.iloc[i, j]) > 0.5 else 'black')
axes[1, 1].set_title('Корреляция признаков\n(X1 и X2 сильно коррелируют!)',
                     fontsize=12, fontweight='bold')
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout()
plt.show()

print(f"Коэффициент β₁ в простой регрессии: {z[0]:.4f}")
print(f"Коэффициент β₁ в множественной регрессии: {beta1_full:.4f}")
print(f"Коэффициент β₁ по теореме FWL: {beta1_fwl:.4f}")
print(f"\nРазница между множественной и FWL: {abs(beta1_full - beta1_fwl):.2e} (должна быть ~0)")

## Что мы увидели на графике

1. Верхний левый (исходные данные): Если мы просто посмотрим на связь Y и X1 (без учёта X2, X3), получим коэффициент ≈ 1.8. Но это смещённая оценка, потому что X1 коррелирует с X2 (см. нижний правый).

2. Верхний правый (FWL): А вот что видит FWL: мы «очистили» Y и X1 от влияния X2 и X3. Остался «чистый» эффект X1 на Y. Коэффициент ≈ 3.0 — это и есть истинное влияние.

3. Нижний левый (сравнение коэффициентов): Простая регрессия дала 1.8, множественная — 3.0, FWL — тоже 3.0. Теорема подтверждена: множественная регрессия и FWL дают одинаковый результат.

4. Нижний правый (корреляция признаков): Почему простая регрессия ошиблась? Потому что X1 и X2 коррелируют (r=0.81). Когда мы не контролируем X2, часть его эффекта «приписывается» X1.

## Ключевой вывод для ML

В машинном обучении признаки часто коррелируют между собой (мультиколлинеарность). Теорема FWL напоминает: коэффициент при признаке должен быть его маргинальный вклад, очищенный от всех остальных.

---

# Итоговое резюме

## Сводная таблица: Статистика → Machine Learning

| Блок | Статистический инструмент | ML-задача | Когда применять |
| :--- | :--- | :--- | :--- |
| **1. F-тест** | Сравнение вложенных моделей | Отбор признаков: «Нужны ли эти фичи?» | Когда хотите добавить/удалить группу признаков |
| **2. Гетероскедастичность** | Тест Бройша-Пагана, робастные SE | Диагностика модели: «Можно ли доверять ДИ?» | Всегда перед построением ДИ для коэффициентов |
| **3. Нормальность остатков** | Тесты KS, AD, JB + QQ-plot | Адекватность модели: «Подходит ли линейная модель?» | После обучения модели, перед интерпретацией |
| **4. Теорема FWL** | Partial regression | Интерпретация: «Что на самом деле измеряет $\beta$?» | При анализе мультиколлинеарности, для explainable AI |

## Ключевые выводы

1. **F-тест** — строгий статистический аналог интуиции «нужен ли этот признак?». Работает для вложенных моделей.

2. **Гетероскедастичность** — если дисперсия ошибок не постоянна, доверительные интервалы лгут. Используйте робастные стандартные ошибки (HC3).

3. **Нормальность остатков** — не доверяйте одному тесту. Для ML-данных с выбросами используйте Anderson-Darling или Jarque-Bera.

4. **Теорема FWL** — коэффициент при признаке — это его маргинальный вклад, очищенный от всех остальных. Связана с Partial Dependence Plots и SHAP.

---